# TC3-82 – Levantar e integrar bases socioeconômicas e financeiras municipais

**Responsável:** Mauricio Cruz

**Sprint:** Fase 2

**Jira:** TC3-82

## Objetivo

Levantar, baixar, documentar e integrar as novas bases socioeconômicas e financeiras municipais que podem enriquecer as análises exploratórias e futuras etapas de modelagem.


### Carrega dados ml-alunos utilizando os passos do TC3_38


## Inicialização independente

Esta célula reproduz a configuração compartilhada do notebook principal. Ela prepara ambiente, caminhos e funções comuns sem alterar a lógica canônica desta etapa.


In [1]:
# =============================================================================
# CONFIGURAÇÕES PARA RODAR O NOTEBOOK
# Execute esta célula primeiro. Esta política não altera a metodologia do fluxo.
# =============================================================================
import os
import sys
import gc
import ctypes
import polars as pl
from datetime import datetime, timezone

# Opções do usuário
PERFIL_RECURSOS = "auto"       # auto | baixo | medio | normal | alto
LIMITE_RAM_PERCENTUAL = None    # None = automático; ou inteiro entre 50 e 90
MAX_THREADS = None              # None = automático; ou inteiro >= 1
USAR_GPU = os.getenv("TC3_USAR_GPU", "1").strip().lower() in {"1", "true", "sim", "yes"}                 # requer WSL2 + RAPIDS/cuML + kernel tc3-gpu

REUTILIZAR_ARTEFATOS = False  # False reconstrói; True aceita caches válidos
SALVAR_ARTEFATOS = True       # False usa somente área temporária da sessão


# Protocolo metodologico compartilhado por todas as fases e modelos.
SEED_GLOBAL = 42
EXECUTAR_SHAP = os.getenv("TC3_EXECUTAR_SHAP", "0").strip().lower() in {"1", "true", "sim", "yes"}
AMOSTRA_SHAP = max(20, int(os.getenv("TC3_AMOSTRA_SHAP", "100")))
AMOSTRA_BACKGROUND_SHAP = max(20, int(os.getenv("TC3_BACKGROUND_SHAP", "50")))
CHAVE_GRUPO_GLOBAL = "chave_escola_ano"
N_SPLITS_CV_GLOBAL = 3
N_ITER_BUSCA_GLOBAL = 10
LIMIAR_CLASSIFICACAO_GLOBAL = 0.50
REPETICOES_PERMUTACAO_GLOBAL = 5
REPETICOES_BOOTSTRAP_GLOBAL = 200

N_JOBS_MODELAGEM_GLOBAL = 1
N_SPLITS_HOLDOUT_GLOBAL = 1
SCORING_PRINCIPAL_GLOBAL = "roc_auc"
VERBOSE_BUSCA_GLOBAL = 2
ERRO_BUSCA_GLOBAL = "raise"
RETORNAR_SCORE_TREINO_GLOBAL = True

# Contrato comum de pre-processamento dos modelos.
IMPUTACAO_NUMERICA_GLOBAL = "median"
IMPUTACAO_CATEGORICA_GLOBAL = "most_frequent"
ONE_HOT_DESCONHECIDO_GLOBAL = "ignore"
ONE_HOT_SPARSO_GLOBAL = False  # matriz densa: evita fallback dos modelos para CPU
SPARSE_THRESHOLD_GLOBAL = 0.0
ESCALONADOR_CENTRALIZAR_GLOBAL = False


# Percentuais de amostragem (0 < valor <= 1). Alterar aqui afeta todo o notebook.
# O teste final permanece integral (100%); as amostras menores servem apenas ao
# desenvolvimento, comparacao de modelos e interpretabilidade.
FRACOES_AMOSTRAGEM = {
    "eda_correlacao": 0.05,
    "validacao_preparacao": 0.02,
    "fase5_modelos": 0.10,
    "fase5_knn": 0.02,  # KNN tem custo de memoria e predicao maior.
    "fase6_desenvolvimento": 0.12,
    "fase6_validacao_interna": 0.25,
    "fase7_desenvolvimento": 1.00,
    "fase7_teste": 1.00,
    "interpretacao": 0.035,
    "bootstrap": 0.17,
}

for _nome, _fracao in FRACOES_AMOSTRAGEM.items():
    if not 0 < _fracao <= 1:
        raise ValueError(f"Fracao invalida em FRACOES_AMOSTRAGEM[{_nome!r}]: {_fracao}")


def tamanho_por_fracao(total, fracao):
    """Converte uma proporcao em quantidade reproduzivel, sem exceder o total."""
    if total < 1:
        raise ValueError("Nao e possivel amostrar uma particao vazia.")
    return min(total, max(1, round(total * fracao)))


def materializar_split_percentual(dados, contrato, nome_split, fracao, seed, materializador):
    """Materializa uma fracao de um split e informa claramente o universo usado."""
    import polars as pl

    coluna_split = contrato["controle_split"]
    total = (
        dados.filter(pl.col(coluna_split) == nome_split)
        .select(pl.len().alias("total"))
        .collect()
        .item()
    )
    quantidade = tamanho_por_fracao(total, fracao)
    X, y = materializador(dados, contrato, nome_split, limit=quantidade, seed=seed)
    print(f"{nome_split}: {quantidade:,}/{total:,} registros ({fracao:.1%})")
    return X, y


def _estado_memoria_sistema():
    """Retorna RAM total e disponível em bytes."""
    valores = {}
    try:
        with open("/proc/meminfo", encoding="utf-8") as arquivo:
            for linha in arquivo:
                chave, valor = linha.split(":", 1)
                valores[chave] = int(valor.strip().split()[0]) * 1024
        return valores["MemTotal"], valores.get("MemAvailable", valores.get("MemFree", 0))
    except (OSError, KeyError, ValueError):
        try:
            return (
                os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE"),
                os.sysconf("SC_AVPHYS_PAGES") * os.sysconf("SC_PAGE_SIZE"),
            )
        except (AttributeError, OSError, ValueError):
            return None, None


RAM_TOTAL_BYTES, RAM_DISPONIVEL_BYTES = _estado_memoria_sistema()
RAM_TOTAL_GB = RAM_TOTAL_BYTES / 1024**3 if RAM_TOTAL_BYTES else 0.0
CPUS_LOGICAS = os.cpu_count() or 1

if RAM_TOTAL_GB and RAM_TOTAL_GB <= 4.5:
    _perfil_detectado = "baixo"
elif RAM_TOTAL_GB and RAM_TOTAL_GB <= 8.5:
    _perfil_detectado = "medio"
elif RAM_TOTAL_GB and RAM_TOTAL_GB <= 16.5:
    _perfil_detectado = "normal"
else:
    _perfil_detectado = "alto"

_perfis = {
    "baixo": (60, 1),
    "medio": (70, min(2, CPUS_LOGICAS)),
    "normal": (75, min(4, CPUS_LOGICAS)),
    "alto": (80, min(8, max(1, CPUS_LOGICAS // 2))),
}
PERFIL_ATIVO = _perfil_detectado if PERFIL_RECURSOS == "auto" else PERFIL_RECURSOS.lower()
if PERFIL_ATIVO not in _perfis:
    raise ValueError("PERFIL_RECURSOS deve ser auto, baixo, medio, normal ou alto.")

_limite_perfil, _threads_perfil = _perfis[PERFIL_ATIVO]
LIMITE_RAM_ATIVO = _limite_perfil if LIMITE_RAM_PERCENTUAL is None else int(LIMITE_RAM_PERCENTUAL)
THREADS_MAXIMAS = _threads_perfil if MAX_THREADS is None else int(MAX_THREADS)
if not 50 <= LIMITE_RAM_ATIVO <= 90:
    raise ValueError("LIMITE_RAM_PERCENTUAL deve ficar entre 50 e 90.")
if THREADS_MAXIMAS < 1:
    raise ValueError("MAX_THREADS deve ser pelo menos 1.")
THREADS_MAXIMAS = min(THREADS_MAXIMAS, CPUS_LOGICAS)

# Definido antes de NumPy, Polars, scikit-learn e cuML serem importados.
for _variavel, _valor in {
    "OMP_NUM_THREADS": str(THREADS_MAXIMAS),
    "OPENBLAS_NUM_THREADS": str(THREADS_MAXIMAS),
    "MKL_NUM_THREADS": str(THREADS_MAXIMAS),
    "NUMEXPR_NUM_THREADS": str(THREADS_MAXIMAS),
    "VECLIB_MAXIMUM_THREADS": str(THREADS_MAXIMAS),
    "POLARS_MAX_THREADS": str(THREADS_MAXIMAS),
    "MALLOC_ARENA_MAX": "2",
}.items():
    os.environ[_variavel] = _valor

HISTORICO_RECURSOS = []
_estado_recursos = {"nivel": None, "threads": None, "contexto_threads": None}


def _aplicar_threads(limite):
    """Limita pools BLAS/OpenMP já carregados; Polars usa o limite inicial."""
    limite = max(1, min(int(limite), THREADS_MAXIMAS))
    if _estado_recursos["threads"] == limite:
        return
    anterior = _estado_recursos.get("contexto_threads")
    if anterior is not None:
        try:
            anterior.__exit__(None, None, None)
        except Exception:
            pass
    try:
        from threadpoolctl import threadpool_limits
        contexto = threadpool_limits(limits=limite)
        contexto.__enter__()
    except Exception:
        contexto = None
    _estado_recursos["contexto_threads"] = contexto
    _estado_recursos["threads"] = limite


def _liberar_memoria_global(agressiva=False):
    """Libera caches transitórios sem apagar variáveis usadas pelo fluxo."""
    shell = get_ipython()
    if shell is not None:
        for nome in ("_", "__", "___"):
            shell.user_ns[nome] = None
    gc.collect()
    modulo_cupy = sys.modules.get("cupy")
    if modulo_cupy is not None:
        try:
            modulo_cupy.get_default_memory_pool().free_all_blocks()
            modulo_cupy.get_default_pinned_memory_pool().free_all_blocks()
        except Exception:
            pass
    if agressiva:
        modulo_plt = sys.modules.get("matplotlib.pyplot")
        if modulo_plt is not None:
            try:
                modulo_plt.close("all")
            except Exception:
                pass
        try:
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except (OSError, AttributeError):
            pass


def _gerenciar_recursos(*_args, **_kwargs):
    """Reduz paralelismo progressivamente e nunca interrompe uma célula."""
    total, disponivel = _estado_memoria_sistema()
    if not total:
        return
    uso = 100.0 * (total - disponivel) / total
    margem = LIMITE_RAM_ATIVO - uso
    if margem > 15:
        nivel, threads, limpar = "normal", THREADS_MAXIMAS, False
    elif margem > 7:
        nivel, threads, limpar = "moderado", max(1, THREADS_MAXIMAS // 2), True
    else:
        nivel, threads, limpar = "conservador", 1, True

    _aplicar_threads(threads)
    if limpar:
        _liberar_memoria_global(agressiva=margem <= 0)

    if nivel != _estado_recursos["nivel"]:
        registro = {
            "instante_utc": datetime.now(timezone.utc).isoformat(),
            "nivel": nivel,
            "threads": threads,
            "ram_usada_percentual": round(uso, 1),
            "ram_disponivel_gb": round(disponivel / 1024**3, 2),
        }
        HISTORICO_RECURSOS.append(registro)
        print(
            f"[recursos] nível={nivel}; threads={threads}; "
            f"RAM usada={uso:.1f}%; livre={disponivel / 1024**3:.1f} GB."
        )
        _estado_recursos["nivel"] = nivel


ipython = get_ipython()
if ipython is not None:
    # Impede callbacks duplicados ao reexecutar a configuração.
    for _atributo in ("_tc3_memory_guard", "_tc3_resource_guard"):
        _guard_anterior = getattr(ipython, _atributo, None)
        if _guard_anterior is not None:
            for _evento, _funcao in _guard_anterior:
                try:
                    ipython.events.unregister(_evento, _funcao)
                except (ValueError, KeyError):
                    pass
    _callbacks_recursos = [
        ("pre_run_cell", _gerenciar_recursos),
        ("post_run_cell", _gerenciar_recursos),
    ]
    for _evento, _funcao in _callbacks_recursos:
        ipython.events.register(_evento, _funcao)
    ipython._tc3_resource_guard = _callbacks_recursos

_gerenciar_recursos()
print(
    "Configuração automática ativa: "
    f"perfil={PERFIL_ATIVO}; RAM={RAM_TOTAL_GB:.1f} GB; CPUs={CPUS_LOGICAS}; "
    f"limite preventivo={LIMITE_RAM_ATIVO}%; threads máximas={THREADS_MAXIMAS}."
)

# Backend GPU: ativado antes de qualquer importação do scikit-learn.
cuda_conda = os.path.join(sys.prefix, "targets", "x86_64-linux")
if os.path.exists(os.path.join(cuda_conda, "include", "cuda.h")):
    os.environ.setdefault("CUDA_PATH", cuda_conda)
if USAR_GPU:
    try:
        from cuml.accel import install
        install(log_level="info")
        import cupy as cp
        dispositivo = cp.cuda.runtime.getDeviceProperties(0)
        nome_gpu = dispositivo["name"].decode() if isinstance(dispositivo["name"], bytes) else str(dispositivo["name"])
        memoria_gpu_gb = cp.cuda.runtime.memGetInfo()[1] / 1024**3
        BACKEND_ML = f"RAPIDS cuML — {nome_gpu} ({memoria_gpu_gb:.1f} GB de VRAM)"
    except Exception as exc:
        raise RuntimeError("GPU/RAPIDS indisponível. Abra no WSL2 e selecione o kernel tc3-gpu.") from exc
else:
    BACKEND_ML = "scikit-learn CPU"
print("Backend de Machine Learning:", BACKEND_ML)

from pathlib import Path
import hashlib
import json
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

def localizar_raiz(inicio: Path) -> Path:
    for candidato in [inicio, *inicio.parents]:
        if (candidato / "dados").is_dir() and (candidato / "notebooks").is_dir():
            return candidato
    raise FileNotFoundError("Execute este notebook dentro do repositório tech_challenge_3.")

PROJECT_ROOT = localizar_raiz(Path.cwd().resolve())
print("Raiz do projeto:", PROJECT_ROOT)


# Contrato único de diretórios. Fontes sempre são lidas da área imutável.
import tempfile

FONTES_ORIGINAIS_DIR = PROJECT_ROOT / "dados" / "fontes_originais"
_TEMP_BASE = PROJECT_ROOT / ".execucao_temporaria"
_TEMP_BASE.mkdir(parents=True, exist_ok=True)
if SALVAR_ARTEFATOS:
    RAIZ_ARTEFATOS = PROJECT_ROOT
    _DIRETORIO_TEMPORARIO = None
else:
    _DIRETORIO_TEMPORARIO = tempfile.TemporaryDirectory(prefix="tc3_", dir=_TEMP_BASE)
    RAIZ_ARTEFATOS = Path(_DIRETORIO_TEMPORARIO.name)

DADOS_PROCESSADOS_DIR = RAIZ_ARTEFATOS / "dados" / "dados_processados"
DADOS_ANALITICOS_DIR = RAIZ_ARTEFATOS / "dados" / "dados_analiticos"
RESULTADOS_DIR = RAIZ_ARTEFATOS / "resultados"
RESULTADOS_FIGURAS_DIR = RESULTADOS_DIR / "figuras"
RESULTADOS_TABELAS_DIR = RESULTADOS_DIR / "tabelas"
RESULTADOS_METRICAS_DIR = RESULTADOS_DIR / "metricas"
RESULTADOS_RELATORIOS_DIR = RESULTADOS_DIR / "relatorios"
MODELOS_DIR = RAIZ_ARTEFATOS / "modelos"

for _diretorio_saida in (
    DADOS_PROCESSADOS_DIR, DADOS_ANALITICOS_DIR, RESULTADOS_FIGURAS_DIR,
    RESULTADOS_TABELAS_DIR, RESULTADOS_METRICAS_DIR,
    RESULTADOS_RELATORIOS_DIR, MODELOS_DIR,
):
    _diretorio_saida.mkdir(parents=True, exist_ok=True)

def deve_recalcular(caminho):
    """True quando o cache deve ser ignorado ou ainda não existe."""
    return not REUTILIZAR_ARTEFATOS or not Path(caminho).exists()


def publicar_parquet_particionado(frame, destino, *, particionar_por):
    """Publica partições sequencialmente e troca o diretório de forma atômica.

    O ``write_parquet(partition_by=...)`` do Polars prepara várias partições em
    paralelo e pode duplicar o pico de RAM de bases já materializadas. Esta
    implementação mantém apenas uma partição adicional em memória por vez.
    """
    import gc
    import shutil

    destino = Path(destino)
    temporario = destino.with_name(destino.name + "__publicacao_tmp")
    anterior = destino.with_name(destino.name + "__publicacao_anterior")
    for caminho in (temporario, anterior):
        if caminho.exists():
            shutil.rmtree(caminho)
    temporario.mkdir(parents=True, exist_ok=True)

    lazy = frame if isinstance(frame, pl.LazyFrame) else frame.lazy()
    schema = lazy.collect_schema()
    if particionar_por not in schema.names():
        raise ValueError(f"Coluna de particionamento ausente: {particionar_por}")

    valores = (
        lazy.select(pl.col(particionar_por).unique())
        .collect(engine="streaming")
        .get_column(particionar_por)
        .to_list()
    )
    valores = sorted(valores, key=lambda valor: (valor is None, str(valor)))
    if not valores:
        raise RuntimeError("Não há valores para particionar a publicação.")

    arquivos_novos = []
    for indice, valor in enumerate(valores):
        valor_hive = "__HIVE_DEFAULT_PARTITION__" if valor is None else str(valor)
        if any(separador in valor_hive for separador in ("/", "\\")):
            raise ValueError(f"Valor de partição inválido: {valor_hive!r}")
        diretorio_particao = temporario / f"{particionar_por}={valor_hive}"
        diretorio_particao.mkdir(parents=True, exist_ok=True)
        arquivo = diretorio_particao / f"part-{indice:05d}.parquet"
        predicado = pl.col(particionar_por).is_null() if valor is None else pl.col(particionar_por) == pl.lit(valor)
        lazy.filter(predicado).sink_parquet(
            arquivo,
            compression="zstd",
            statistics=True,
            row_group_size=100_000,
            mkdir=True,
            engine="streaming",
        )
        if not arquivo.is_file() or arquivo.stat().st_size == 0:
            raise RuntimeError(f"Partição não foi gravada corretamente: {arquivo}")
        arquivos_novos.append(arquivo)
        gc.collect()

    if destino.exists():
        destino.replace(anterior)
    try:
        try:
            temporario.replace(destino)
        except OSError:
            # Volumes Docker montados do Windows podem bloquear rename de diretórios.
            # O fallback copia a publicação pronta e remove a área temporária.
            shutil.copytree(temporario, destino)
            shutil.rmtree(temporario)
    except Exception:
        if destino.exists():
            shutil.rmtree(destino)
        if anterior.exists():
            anterior.replace(destino)
        raise
    if anterior.exists():
        shutil.rmtree(anterior)
    return len(arquivos_novos)


def salvar_figura_eda(figura, nome):
    """Exporta uma figura exploratória sem alterar sua exibição no notebook."""
    destino = RESULTADOS_FIGURAS_DIR / "fluxo_exploratorio" / nome
    destino.parent.mkdir(parents=True, exist_ok=True)
    figura.savefig(destino, dpi=180, bbox_inches="tight")
    return destino

print("Fontes originais:", FONTES_ORIGINAIS_DIR)
print("Destino dos artefatos:", RAIZ_ARTEFATOS)
print("Reutilizar artefatos:", REUTILIZAR_ARTEFATOS, "| Salvar:", SALVAR_ARTEFATOS)

# Imports compartilhados que, no fluxo completo, são carregados por etapas anteriores.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display


[recursos] nível=normal; threads=4; RAM usada=40.7%; livre=9.2 GB.
Configuração automática ativa: perfil=normal; RAM=15.5 GB; CPUs=20; limite preventivo=75%; threads máximas=4.
[cuml.accel] Accelerator installed.
Backend de Machine Learning: RAPIDS cuML — NVIDIA GeForce RTX 3050 6GB Laptop GPU (6.0 GB de VRAM)
Raiz do projeto: /workspace
Fontes originais: /workspace/dados/fontes_originais
Destino dos artefatos: /workspace
Reutilizar artefatos: False | Salvar: True


In [2]:
import polars as pl
import zipfile
from pathlib import Path


In [3]:
# A fonte compactada é somente leitura; sua extração é regenerável.
ZIP_PATH = FONTES_ORIGINAIS_DIR / "principal" / "ml_aluno.zip"
EXTRACT_PATH = DADOS_PROCESSADOS_DIR / "fontes_extraidas" / "principal"
EXTRACT_PATH.mkdir(parents=True, exist_ok=True)

if not (EXTRACT_PATH / "ml_aluno").exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)

# Caminho local da feature table extraída
BASE_PATH = EXTRACT_PATH / "ml_aluno"

# Partições da Gold
PATH_2023 = BASE_PATH / "ano2023"
PATH_2024 = BASE_PATH / "ano2024"

arquivos_2023 = sorted(PATH_2023.glob("*.parquet"))
arquivos_2024 = sorted(PATH_2024.glob("*.parquet"))

df_2023 = pl.concat(
    [pl.read_parquet(arq) for arq in arquivos_2023]
)

df_2024 = pl.concat(
    [pl.read_parquet(arq) for arq in arquivos_2024]
)

# Consolidação da base
df = pl.concat([df_2023, df_2024])


### Carrega arquivos complementares


In [4]:
PATH_COMPLEMENTAR = PROJECT_ROOT / "dados" / "fontes_originais" / "complementar" 

#Bolsa Familia
df_bolsa_bronze = pl.read_csv(PATH_COMPLEMENTAR / "base_dados_bolsa_familia_2023_2024.csv")

#Fundeb
df_fundeb_bronze = pl.read_csv(PATH_COMPLEMENTAR / "base_dados_fundeb_indicador_municipal_2023_2024.csv")

#Receitas Orcamentarias
df_receitas_bronze = pl.read_csv(PATH_COMPLEMENTAR / "base_de_dados_municipio_receitas_orcamentarias_2023_2024.csv")

#PIB
df_pib_bronze = pl.read_csv(
    PATH_COMPLEMENTAR / "base_de_dados_PIB_Municipios_2022_2023.csv",
    separator=";",           # o arquivo usa ;
    decimal_comma=True        # números com vírgula
)

#INSE
df_inse_bronze = pl.read_excel(PATH_COMPLEMENTAR / "INSE_2023_escolas.xlsx", sheet_name="INSE_ESC_2023", engine="openpyxl")  # ou sheet_name=0 (primeira aba)


#Censo Escolar
ZIP_PATH_CENSO = PATH_COMPLEMENTAR / "microdados_ed_basica_2023.zip"
CSV_PATH_CENSO = PATH_COMPLEMENTAR / "microdados_ed_basica_2023.csv"

if ZIP_PATH_CENSO.exists() and not CSV_PATH_CENSO.exists():
    with zipfile.ZipFile(ZIP_PATH_CENSO, "r") as zip_ref:
        zip_ref.extractall(PATH_COMPLEMENTAR)

if CSV_PATH_CENSO.exists():
    df_censo_bronze = pl.read_csv(
        CSV_PATH_CENSO,
        separator=";",
        decimal_comma=True,
        encoding="utf8-lossy",
        schema_overrides={"CO_ORGAO_REGIONAL": pl.String},
        infer_schema_length=10_000
    )
else:
    print("Arquivo do censo não encontrado.")


### Padronizar os nomes das colunas


In [5]:
def padronizar_colunas(df_origem, renomeacoes=None):
    """Renomeia cada coluna uma única vez, evitando referências ao esquema anterior."""
    renomeacoes = renomeacoes or {}
    mapa = {coluna: renomeacoes.get(coluna, coluna.lower()) for coluna in df_origem.columns}
    destinos = list(mapa.values())
    duplicadas = sorted({nome for nome in destinos if destinos.count(nome) > 1})
    if duplicadas:
        raise ValueError(f"A padronização produziria colunas duplicadas: {duplicadas}")
    return df_origem.rename(mapa)

df_bolsa_silver = padronizar_colunas(
    df_bolsa_bronze,
    {'ano_referencia': 'ano', 'total_valor': 'valor_bolsa_familia'},
)

df_fundeb_silver = padronizar_colunas(
    df_fundeb_bronze,
    {'total_valor': 'valor_fundeb'},
)

df_receitas_silver = padronizar_colunas(
    df_receitas_bronze,
    {'total_valor': 'valor_receitas'},
)

df_pib_silver = padronizar_colunas(
    df_pib_bronze,
    {'Ano': 'ano', 'UF': 'sigla_uf', 'Código do Município': 'id_municipio', 'Produto Interno Bruto, a preços correntes': 'valor_pib', 'Produto Interno Bruto per capita, a preços correntes': 'valor_pib_per_capita'},
)

df_inse_silver = padronizar_colunas(
    df_inse_bronze,
    {'NU_ANO_SAEB': 'ano', 'SG_UF': 'sigla_uf', 'CO_MUNICIPIO': 'id_municipio'},
)

df_censo_silver = padronizar_colunas(
    df_censo_bronze,
    {'NU_ANO_CENSO': 'ano', 'SG_UF': 'sigla_uf', 'CO_MUNICIPIO': 'id_municipio'},
)


### Padronizar os tipos das colunas


In [6]:
df = df.with_columns(pl.col("id_municipio", "ano_referencia").cast(pl.Int64))
df_bolsa_silver = df_bolsa_silver.with_columns(pl.col("valor_bolsa_familia").cast(pl.Float64, strict=False))
df_fundeb_silver = df_fundeb_silver.with_columns(pl.col("valor_fundeb").cast(pl.Float64, strict=False))
df_receitas_silver = df_receitas_silver.with_columns(pl.col("valor_receitas").cast(pl.Float64, strict=False))
for nome in ["valor_pib", "valor_pib_per_capita"]:
    df_pib_silver = df_pib_silver.with_columns(
        pl.col(nome).cast(pl.String).str.replace_all("\\.", "").str.replace(",", ".").str.strip_chars().cast(pl.Float64, strict=False)
    )
colunas_com_codigo = [c for c in df_censo_silver.columns if df_censo_silver[c].cast(pl.String).eq("0MI11").any()]
df_censo_silver = df_censo_silver.with_columns([pl.col(c).cast(pl.String) for c in colunas_com_codigo])


### Unir todas as bases

#### Todas as fontes são consolidadas no nível município/ano antes do `join`. INSE e Censo chegam no nível escolar e são agregados para proteger a granularidade adotada e impedir a explosão de linhas.


In [7]:

# Todas as tabelas desta etapa já são DataFrames Polars; não há conversões ou cópias redundantes.
df_aluno_silver = df

chaves_municipio = ["ano", "id_municipio", "sigla_uf"]

# O INSE é escolar na origem. A média e os percentuais são ponderados
# pela quantidade de alunos representados em cada escola.
colunas_inse_ponderadas = [
    "media_inse",
    "pc_nivel_1", "pc_nivel_2", "pc_nivel_3", "pc_nivel_4",
    "pc_nivel_5", "pc_nivel_6", "pc_nivel_7", "pc_nivel_8",
]
df_inse_municipal = (
    df_inse_silver
    .with_columns(
        pl.col("qtd_alunos_inse").cast(pl.Float64, strict=False),
        *[pl.col(col).cast(pl.Float64, strict=False) for col in colunas_inse_ponderadas],
    )
    .group_by(chaves_municipio)
    .agg(
        pl.col("id_escola").n_unique().alias("qtd_escolas_inse"),
        pl.col("qtd_alunos_inse").sum().alias("qtd_alunos_inse"),
        *[
            (
                (pl.col(col) * pl.col("qtd_alunos_inse")).sum()
                / pl.col("qtd_alunos_inse").sum()
            ).alias(col)
            for col in colunas_inse_ponderadas
        ],
    )
)

# O Censo também é escolar na origem. Totais são somados e indicadores
# binários viram a proporção de escolas do município com cada recurso.
colunas_censo_totais = [
    "qt_mat_bas", "qt_doc_bas", "qt_tur_bas"
]
colunas_censo_indicadores = [
    "in_biblioteca_sala_leitura", "in_laboratorio_informatica", "in_quadra_esportes",
    "in_agua_rede_publica", "in_energia_rede_publica",
    "in_esgoto_rede_publica", "in_internet", "in_banda_larga",
]
df_censo_municipal = (
    df_censo_silver
    .with_columns(
        *[
            pl.col(col).cast(pl.Float64, strict=False)
            for col in colunas_censo_totais + colunas_censo_indicadores
        ]
    )
    .group_by(chaves_municipio)
    .agg(
        pl.col("co_entidade").n_unique().alias("qtd_escolas_censo"),
        *[pl.col(col).sum().alias(col) for col in colunas_censo_totais],
        *[
            pl.col(col).mean().alias(col)
            for col in colunas_censo_indicadores
        ],
    )
)

# Política temporal explícita:
# - Bolsa Família, Fundeb e receitas: correspondência anual exata.
# - PIB: disponibilidade defasada em um ano (2022 -> alunos 2023; 2023 -> alunos 2024).
# - INSE e Censo Escolar: retrato contextual de 2023 reutilizado nos dois anos,
#   sem interpretar a repetição como evolução temporal.
df_pib_lag = (
    df_pib_silver
    .with_columns((pl.col("ano") + 1).alias("ano_referencia_join"))
    .drop("ano")
)
df_inse_snapshot = df_inse_municipal.filter(pl.col("ano") == 2023).drop("ano")
df_censo_snapshot = df_censo_municipal.filter(pl.col("ano") == 2023).drop("ano")

chaves_aluno_ano = ["ano_referencia", "id_municipio", "sigla_uf"]
chaves_fonte_ano = ["ano", "id_municipio", "sigla_uf"]
chaves_territorio = ["id_municipio", "sigla_uf"]

qtd_linhas_antes_dos_joins = df_aluno_silver.height
df_aluno_silver = (
    df_aluno_silver
    .join(df_bolsa_silver, left_on=chaves_aluno_ano, right_on=chaves_fonte_ano, how="left", validate="m:1")
    .join(df_fundeb_silver, left_on=chaves_aluno_ano, right_on=chaves_fonte_ano, how="left", validate="m:1")
    .join(df_receitas_silver, left_on=chaves_aluno_ano, right_on=chaves_fonte_ano, how="left", validate="m:1")
    .join(
        df_pib_lag,
        left_on=["ano_referencia", "id_municipio", "sigla_uf"],
        right_on=["ano_referencia_join", "id_municipio", "sigla_uf"],
        how="left",
        validate="m:1",
    )
    .join(df_inse_snapshot, left_on=chaves_territorio, right_on=chaves_territorio, how="left", validate="m:1")
    .join(df_censo_snapshot, left_on=chaves_territorio, right_on=chaves_territorio, how="left", validate="m:1")
)

assert df_aluno_silver.height == qtd_linhas_antes_dos_joins, (
    "A quantidade de alunos mudou durante os joins: "
    f"{qtd_linhas_antes_dos_joins:,} -> {df_aluno_silver.height:,}"
)
print("Shape após integração temporal explícita:", df_aluno_silver.shape)


Shape após integração temporal explícita: (3867999, 49)
[recursos] nível=conservador; threads=1; RAM usada=71.4%; livre=4.4 GB.


### Auditoria temporal das fontes complementares

As fontes não possuem a mesma janela de publicação. A tabela a seguir torna explícito o tratamento adotado e mede a cobertura efetiva por ano. Isso evita confundir “mais arquivos” com “mais informação comparável” e impede o uso acidental de informação futura.


In [8]:
politica_temporal_fontes = pl.DataFrame({
    "fonte": ["Bolsa Família", "Fundeb", "Receitas", "PIB", "INSE", "Censo Escolar"],
    "anos_disponiveis": ["2023-2024", "2023-2024", "2023-2024", "2022-2023", "2023", "2023"],
    "regra": [
        "correspondência anual exata",
        "correspondência anual exata",
        "correspondência anual exata",
        "defasagem de um ano: 2022→2023 e 2023→2024",
        "snapshot contextual 2023; sem interpretação temporal",
        "snapshot contextual 2023; sem interpretação temporal",
    ],
})
display(politica_temporal_fontes)

colunas_por_fonte = {
    "Bolsa Família": "valor_bolsa_familia",
    "Fundeb": "valor_fundeb",
    "Receitas": "valor_receitas",
    "PIB": "valor_pib",
    "INSE": "media_inse",
    "Censo Escolar": "qtd_alunos_inse",
}
linhas_cobertura = []
for ano in sorted(df_aluno_silver["ano_referencia"].unique().to_list()):
    recorte = df_aluno_silver.filter(pl.col("ano_referencia") == ano)
    for fonte, coluna in colunas_por_fonte.items():
        if coluna not in recorte.columns:
            continue
        preenchidos = recorte.select(pl.col(coluna).is_not_null().sum()).item()
        linhas_cobertura.append({
            "ano_referencia": int(ano),
            "fonte": fonte,
            "registros": recorte.height,
            "preenchidos": int(preenchidos),
            "cobertura_pct": round(100.0 * preenchidos / max(recorte.height, 1), 2),
        })
auditoria_cobertura_fontes = pl.DataFrame(linhas_cobertura).sort("fonte", "ano_referencia")
display(auditoria_cobertura_fontes)


[recursos] nível=moderado; threads=2; RAM usada=67.5%; livre=5.0 GB.


fonte,anos_disponiveis,regra
str,str,str
"""Bolsa Família""","""2023-2024""","""correspondência anual exata"""
"""Fundeb""","""2023-2024""","""correspondência anual exata"""
"""Receitas""","""2023-2024""","""correspondência anual exata"""
"""PIB""","""2022-2023""","""defasagem de um ano: 2022→2023…"
"""INSE""","""2023""","""snapshot contextual 2023; sem …"
"""Censo Escolar""","""2023""","""snapshot contextual 2023; sem …"


ano_referencia,fonte,registros,preenchidos,cobertura_pct
i64,str,i64,i64,f64
2023,"""Bolsa Família""",1747439,1747439,100.0
2024,"""Bolsa Família""",2120560,2120560,100.0
2023,"""Censo Escolar""",1747439,1746238,99.93
2024,"""Censo Escolar""",2120560,2119431,99.95
2023,"""Fundeb""",1747439,1744474,99.83
…,…,…,…,…
2024,"""INSE""",2120560,2119431,99.95
2023,"""PIB""",1747439,1747439,100.0
2024,"""PIB""",2120560,2120560,100.0


In [9]:
print(f"Shape: {df_aluno_silver.shape}")
df_aluno_silver.glimpse()
df_aluno_silver.head()


Shape: (3867999, 49)
Rows: 3867999
Columns: 49
$ id_aluno                                           <str> '31062133', '15078120', '43090743', '42097998', '43072338', '33048792', '24009445', '33014166', '11010561', '32035154'
$ id_escola                                          <str> '60021550', '60001463', '60037407', '60035930', '60037775', '60024671', '60010376', '60023928', '60000036', '60022642'
$ id_municipio                                       <i64> 3136900, 1506005, 4305108, 4219507, 4309209, 3302700, 2401305, 3300704, 1100205, 3204708
$ nome_municipio                                     <str> 'Juruaia', 'Prainha', 'Caxias do Sul', 'Xanxerê', 'Gravataí', 'Maricá', 'Campo Grande', 'Cabo Frio', 'Porto Velho', 'São Gabriel da Palha'
$ sigla_uf                                           <str> 'MG', 'PA', 'RS', 'SC', 'RS', 'RJ', 'RN', 'RJ', 'RO', 'ES'
$ ano_referencia                                     <i64> 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023
$ serie        

id_aluno,id_escola,id_municipio,nome_municipio,sigla_uf,ano_referencia,serie,rede,rede_descricao,caderno,presenca,preenchimento_caderno,proficiencia,peso_aluno,taxa_alfabetizacao_municipio_ano_anterior,media_portugues_municipio_ano_anterior,dataset_split,alfabetizado,processed_at,valor_bolsa_familia,valor_fundeb,valor_receitas,nome uf,nome do município,valor_pib,valor_pib_per_capita,qtd_escolas_inse,qtd_alunos_inse,media_inse,pc_nivel_1,pc_nivel_2,pc_nivel_3,pc_nivel_4,pc_nivel_5,pc_nivel_6,pc_nivel_7,pc_nivel_8,qtd_escolas_censo,qt_mat_bas,qt_doc_bas,qt_tur_bas,in_biblioteca_sala_leitura,in_laboratorio_informatica,in_quadra_esportes,in_agua_rede_publica,in_energia_rede_publica,in_esgoto_rede_publica,in_internet,in_banda_larga
str,str,i64,str,str,i64,i32,i32,str,str,i32,i32,f64,f64,f64,f64,str,i32,datetime[ns],f64,f64,f64,str,str,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""31062133""","""60021550""",3136900,"""Juruaia""","""MG""",2023,2,3,"""Municipal""","""6""",1,1,731.2356,1.322718,null,null,"""treino""",0,2026-09-01 18:03:53.794418,2.312976e6,4.0045e7,4.9867e8,"""Minas Gerais""","""Juruaia""",3.17184005e8,28616.38,4,365.0,5.18189,0.0,3.552137,13.774877,21.857534,27.463945,22.103781,11.241507,0.0,7,2143.0,114.0,104.0,0.428571,0.142857,0.571429,0.857143,1.0,0.714286,1.0,1.0
"""15078120""","""60001463""",1506005,"""Prainha""","""PA""",2023,2,3,"""Municipal""","""6""",1,1,732.783236,1.3249731,null,null,"""teste""",0,2026-09-01 18:03:53.794418,5.7458393e7,2.2302e8,null,"""Pará""","""Prainha""",4.94695747e8,13904.93,25,1556.0,3.887114,15.029878,40.511697,23.331555,12.725039,5.014814,2.360617,1.030463,0.0,124,12179.0,597.0,658.0,0.09434,0.009434,0.075472,0.150943,0.509434,0.018868,0.367925,0.615385
"""43090743""","""60037407""",4305108,"""Caxias do Sul""","""RS""",2023,2,3,"""Municipal""","""6""",1,1,755.341541,1.3818684,null,null,"""treino""",1,2026-09-01 18:03:53.794418,8.7984651e7,1.4014e9,2.5271e10,"""Rio Grande do Sul""","""Caxias do Sul""",3.4342e10,74118.89,124,13823.0,5.524075,0.061658,1.770355,6.163758,14.475369,24.831173,27.450799,23.430008,1.816964,420,94962.0,5877.0,4749.0,0.68642,0.385185,0.338272,0.995062,0.997531,0.958025,0.997531,0.94802
"""42097998""","""60035930""",4219507,"""Xanxerê""","""SC""",2023,2,3,"""Municipal""","""6""",1,1,711.643712,2.0121879,null,null,"""treino""",0,2026-09-01 18:03:53.794418,6.345159e6,2.1612e8,2.1373e9,"""Santa Catarina""","""Xanxerê""",2.8859e9,55921.22,20,1584.0,5.492967,0.0,1.978005,6.863251,15.859703,24.422923,25.647096,23.055972,2.174122,53,11783.0,881.0,649.0,0.596154,0.423077,0.442308,0.884615,1.0,0.326923,1.0,0.980769
"""43072338""","""60037775""",4309209,"""Gravataí""","""RS""",2023,2,3,"""Municipal""","""7""",0,0,null,null,null,null,"""treino""",0,2026-09-01 18:03:53.794418,1.10156423e8,7.1742e8,1.0518e10,"""Rio Grande do Sul""","""Gravataí""",1.3006e10,49064.69,85,8578.0,5.401837,0.244155,2.778186,7.99628,17.907062,26.069724,23.591075,19.782966,1.631288,185,54762.0,3581.0,2724.0,0.631285,0.24581,0.340782,0.944134,0.994413,0.759777,0.994413,0.94382


### Salvar a base consolidada

O dataset abaixo mantém uma linha por aluno, reúne as variáveis municipais disponíveis para o respectivo ano e grava os arquivos Parquet em partições por UF.


In [10]:
OUTPUT_DIR = (
    DADOS_PROCESSADOS_DIR / "consolidado" / "base_alunos_municipios_por_uf"
)

arquivos_existentes = list(OUTPUT_DIR.rglob("*.parquet"))
if deve_recalcular(OUTPUT_DIR) or not arquivos_existentes:
    quantidade_publicada = publicar_parquet_particionado(
        df_aluno_silver, OUTPUT_DIR, particionar_por="sigla_uf"
    )
    print(f"Base consolidada recalculada: {quantidade_publicada} arquivo(s).")
else:
    print(f"Base consolidada reutilizada: {len(arquivos_existentes)} arquivo(s).")

arquivos_parquet = list(OUTPUT_DIR.rglob("*.parquet"))
tamanho_mb = sum(arq.stat().st_size for arq in arquivos_parquet) / (1024 ** 2)
print(f"Base consolidada particionada salva em: {OUTPUT_DIR}")
print(f"Linhas: {df_aluno_silver.height:,} | Colunas: {df_aluno_silver.width:,}")
print(f"Partições/arquivos: {len(arquivos_parquet)}")
print(f"Tamanho: {tamanho_mb:,.2f} MB")


Base consolidada recalculada: 26 arquivo(s).
Base consolidada particionada salva em: /workspace/dados/dados_processados/consolidado/base_alunos_municipios_por_uf
Linhas: 3,867,999 | Colunas: 49
Partições/arquivos: 26
Tamanho: 200.92 MB


<a id="fase-3"></a>

# Fase 3 — análise exploratória
